In [ ]:
!pip install git+https://github.com/tensorflow/examples.git

  Cloning https://github.com/tensorflow/examples.git to /tmp/pip-req-build-uqhl2411
  Running command git clone --filter=blob:none --quiet https://github.com/tensorflow/examples.git /tmp/pip-req-build-uqhl2411
  Resolved https://github.com/tensorflow/examples.git to commit 5c3457ae6f82453580df0800511033472c3dc584
  Preparing metadata (setup.py) ... done
  Created wheel for tensorflow-examples: filename=tensorflow_examples-0.1746028588.526394427792345445866534157219010350645891679620-py3-none-any.whl size=301651 sha256=4e424e87062d0cff68847ab7efba8b9b052a7ecf9aa5588e16fb59dd6dc19131
  Stored in directory: /tmp/pip-ephem-wheel-cache-lgj2aiav/wheels/91/9b/e8/6ae2ecc930bd726c578e35b313e987a687bc5ce03c3a42c2d5
Successfully built tensorflow-examples


In [ ]:
import tensorflow as tf

In [ ]:
# import tensorflow_datasets as tfds
from tensorflow_examples.models.pix2pix import pix2pix

import os
import time
import matplotlib.pyplot as plt
from IPython.display import clear_output

AUTOTUNE = tf.data.AUTOTUNE

In [1]:
import cv2
import numpy as np
import albumentations as A
import kagglehub
from tqdm import tqdm

In [2]:
# Download latest version
path = kagglehub.dataset_download("njayadithya/rgb-anon-trainvaltest")

print("Path to dataset files:", path)

100%|██████████| 15.6G/15.6G [02:50<00:00, 98.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/njayadithya/rgb-anon-trainvaltest/versions/1


In [ ]:
path = kagglehub.dataset_download("nguyenhuuduck16hcm/tt100k")

print("Path to dataset files:", path)

100%|██████████| 17.9G/17.9G [03:47<00:00, 84.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nguyenhuuduck16hcm/tt100k/versions/1


## Input Pipeline

In [ ]:
# 1. Настройки путей
ACDC_BASE = "/root/.cache/kagglehub/datasets/njayadithya/rgb-anon-trainvaltest/versions/1"
TT100K_BASE = "/root/.cache/kagglehub/datasets/nguyenhuuduck16hcm/tt100k/versions/1"
TARGET_SIZE = (256, 256)  # Размер как в оригинальном CycleGAN
BATCH_SIZE = 4  # Уменьшите, если всё равно получаете OOM

In [ ]:
# 2. Генератор для потоковой загрузки
def image_generator(folder_path, max_images=5000, is_weather=False):
    """Постепенно загружает изображения, не нагружая RAM"""
    files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))][:max_images]

    for file in tqdm(files, desc=f"Обработка {folder_path}"):
        img_path = os.path.join(folder_path, file)
        img = cv2.imread(img_path)

        if img is None:
            continue

        # Ресайз + нормализация
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, TARGET_SIZE)
        img = (img / 127.5) - 1.0  # [-1, 1]

        # Аугментация только для обычных изображений
        if not is_weather and np.random.rand() > 0.5:
            img = augment_image(img)

        yield img

Old

In [ ]:
# Пути к конкретным подпапкам
ACDC_RAIN_TRAIN = os.path.join(ACDC_BASE, "rgb_anon", "rain", "train")
ACDC_SNOW_TRAIN = os.path.join(ACDC_BASE, "rgb_anon", "snow", "train")
TT100K_TRAIN = os.path.join(TT100K_BASE, "tt100k_2021", "train")

Old

In [ ]:
# Проверяем существование ключевых папок
required_folders = [
    ACDC_RAIN_TRAIN,
    ACDC_SNOW_TRAIN,
    # os.path.join(TT100K_TRAIN, "images")
]

for folder in required_folders:
    if not os.path.exists(folder):
        print(f"ОШИБКА: Папка {folder} не найдена!")
        raise FileNotFoundError(f"Отсутствует {folder}")

In [ ]:
# 3. Аугментация с контролем памяти
def augment_image(img):
    """Безопасная аугментация"""
    img_uint8 = ((img + 1) * 127.5).astype(np.uint8)
    transformed = A.ReplayCompose([
        A.HorizontalFlip(p=0.3),
        A.RandomBrightnessContrast(p=0.2)
    ])(image=img_uint8)
    return (transformed["image"] / 127.5) - 1.0

Old

In [ ]:
# 2. Аугментации (добавим больше вариантов погоды)
weather_transform = A.Compose([
    A.HorizontalFlip(p=0.3),
    A.RandomBrightnessContrast(p=0.3),
    A.RandomRain(p=0.5),          # Дождь
    A.RandomSnow(p=0.3),          # Снег
    A.RandomFog(p=0.2),           # Туман
    A.RandomSunFlare(p=0.1)       # Солнечные блики
])

In [ ]:
# 4. Создаем tf.data.Dataset напрямую из генераторов
def create_datasets(acdc_path, tt100k_path, max_images=5000):
    # Для ACDC (плохая погода)
    acdc_gen = lambda: image_generator(acdc_path, max_images, is_weather=True)
    acdc_ds = tf.data.Dataset.from_generator(
        acdc_gen,
        output_signature=tf.TensorSpec(shape=TARGET_SIZE + (3,), dtype=tf.float32)
    )

    # Для TT100K (обычная погода)
    tt100k_gen = lambda: image_generator(tt100k_path, max_images)
    tt100k_ds = tf.data.Dataset.from_generator(
        tt100k_gen,
        output_signature=tf.TensorSpec(shape=TARGET_SIZE + (3,), dtype=tf.float32)
    )

    # Объединяем и батчим
    return tf.data.Dataset.zip((tt100k_ds, acdc_ds)).batch(BATCH_SIZE).prefetch(2)

In [ ]:
# 5. Использование
ACDC_TRAIN = "/root/.cache/kagglehub/datasets/njayadithya/rgb-anon-trainvaltest/versions/1/rgb_anon/rain/train"
TT100K_TRAIN = "/root/.cache/kagglehub/datasets/nguyenhuuduck16hcm/tt100k/versions/1/tt100k_2021/train"

train_dataset = create_datasets(ACDC_TRAIN, TT100K_TRAIN, max_images=1000)  # Начните с малого

In [ ]:
# 7. Основной процесс
print("Загрузка ACDC (плохая погода)...")
weather_type = "rain"  # Или "snow" для снега
acdc_images = load_acdc_weather(weather_type)

print("Загрузка TT100K (хорошая погода)...")
tt100k_images = load_tt100k_normal(max_images=4000)

# Проверка размеров
print(f"ACDC images: {len(acdc_images)}")
print(f"TT100K images: {len(tt100k_images)}")

if len(acdc_images) == 0 or len(tt100k_images) == 0:
    raise ValueError("Один из датасетов пуст! Проверьте пути и загрузку.")

Загрузка ACDC (плохая погода)...
Загрузка из: /root/.cache/kagglehub/datasets/njayadithya/rgb-anon-trainvaltest/versions/1/rgb_anon/rain/train
Обработка папки: GP010400
Обработка папки: GP020400
Обработка папки: GP020571
Обработка папки: GP020402
Обработка папки: GOPR0402
Обработка папки: GP010402
Обработка папки: GP030400
Обработка папки: GOPR0400
Загружено изображений: 400
Загрузка TT100K (хорошая погода)...
Загрузка из: /root/.cache/kagglehub/datasets/nguyenhuuduck16hcm/tt100k/versions/1/tt100k_2021/train
Загружено изображений: 4000


In [ ]:
# Балансируем датасеты
weather_type = "rain"  # Или "snow"
min_len = min(len(acdc_images), len(tt100k_images))
acdc_images = acdc_images[:min_len]
tt100k_images = tt100k_images[:min_len]

print(f"Итоговый размер: {min_len} пар изображений")

# Создаем датасет как в оригинальном CycleGAN
train_dataset = create_dataset(tt100k_images, acdc_images)

NameError: name 'acdc_images' is not defined

In [ ]:
# 8. Проверка
for normal, weather in train_dataset.take(1):
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    plt.imshow((normal[0].numpy() + 1)/2)
    plt.title("Обычная погода (TT100K)")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.imshow((weather[0].numpy() + 1)/2)
    plt.title("Плохая погода (ACDC)")
    plt.axis('off')

    plt.show()

Обработка /root/.cache/kagglehub/datasets/nguyenhuuduck16hcm/tt100k/versions/1/tt100k_2021/train:   0%|          | 0/1000 [00:00<?, ?it/s]
Обработка /root/.cache/kagglehub/datasets/njayadithya/rgb-anon-trainvaltest/versions/1/rgb_anon/rain/train: 0it [00:00, ?it/s]
Обработка /root/.cache/kagglehub/datasets/nguyenhuuduck16hcm/tt100k/versions/1/tt100k_2021/train:   0%|          | 0/1000 [00:00<?, ?it/s]


In [ ]:
def random_jitter(image):
    """Добавляет случайный crop + resize для увеличения разнообразия"""
    # Убедимся, что работаем с отдельным изображением (без батча)
    if len(image.shape) == 4:
        image = tf.squeeze(image, axis=0)

    # Resize и crop
    image = tf.image.resize(image, [286, 286], method=tf.image.ResizeMethod.NEAREST_NEIGHBOR)
    image = tf.image.random_crop(image, size=[256, 256, 3])
    image = tf.image.random_flip_left_right(image)

    # Добавляем размерность батча обратно, если нужно
    if len(image.shape) == 3:
        image = tf.expand_dims(image, axis=0)
    return image

In [ ]:
# normalizing the images to [-1, 1]
def normalize(image):
  image = tf.cast(image, tf.float32)
  image = (image / 127.5) - 1
  return image

In [ ]:
def preprocess_image_train(image, weather_image):
    """Обрабатывает пару: обычное изображение + погодное"""
    # Удаляем размерность батча для обработки
    image = tf.squeeze(image, axis=0)
    weather_image = tf.squeeze(weather_image, axis=0)

    # Применяем аугментации
    image = random_jitter(image)
    weather_image = random_jitter(weather_image)

    # Нормализация и возврат правильной формы
    return (
        tf.expand_dims(normalize(image), axis=0),
        tf.expand_dims(normalize(weather_image), axis=0)
    )

In [ ]:
def preprocess_image_test(image, label):
  image = normalize(image)
  return image

In [ ]:
sample_normal, sample_weather = next(iter(train_dataset))

plt.subplot(121)
plt.title('Обычная сцена')
plt.imshow(sample_normal[0] * 0.5 + 0.5)  # [-1,1] -> [0,1]

plt.subplot(122)
plt.title('Сцена с дождём/снегом')
plt.imshow(sample_weather[0] * 0.5 + 0.5)

Обработка /root/.cache/kagglehub/datasets/nguyenhuuduck16hcm/tt100k/versions/1/tt100k_2021/train:   0%|          | 0/1000 [00:00<?, ?it/s]
Обработка /root/.cache/kagglehub/datasets/njayadithya/rgb-anon-trainvaltest/versions/1/rgb_anon/rain/train: 0it [00:00, ?it/s]
Обработка /root/.cache/kagglehub/datasets/nguyenhuuduck16hcm/tt100k/versions/1/tt100k_2021/train:   0%|          | 0/1000 [00:00<?, ?it/s]


StopIteration: 

## Import and reuse the Pix2Pix models

In [ ]:
generator_g = pix2pix.unet_generator(3, norm_type='instancenorm')
generator_f = pix2pix.unet_generator(3, norm_type='instancenorm')

discriminator_x = pix2pix.discriminator(norm_type='instancenorm', target=False)
discriminator_y = pix2pix.discriminator(norm_type='instancenorm', target=False)

## Loss functions

In [ ]:
LAMBDA = 10

In [ ]:
loss_obj = tf.keras.losses.BinaryCrossentropy(from_logits=True)

In [ ]:
def discriminator_loss(real, generated):
  real_loss = loss_obj(tf.ones_like(real), real)

  generated_loss = loss_obj(tf.zeros_like(generated), generated)

  total_disc_loss = real_loss + generated_loss

  return total_disc_loss * 0.5

In [ ]:
def generator_loss(generated):
  return loss_obj(tf.ones_like(generated), generated)

In [ ]:
def calc_cycle_loss(real_image, cycled_image):
  loss1 = tf.reduce_mean(tf.abs(real_image - cycled_image))

  return LAMBDA * loss1

In [ ]:
def identity_loss(real_image, same_image):
  loss = tf.reduce_mean(tf.abs(real_image - same_image))
  return LAMBDA * 0.5 * loss

In [ ]:
generator_g_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
generator_f_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

discriminator_x_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_y_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)

## Checkpoints

In [ ]:
checkpoint_path = "./checkpoints/train"

ckpt = tf.train.Checkpoint(generator_g=generator_g,
                           generator_f=generator_f,
                           discriminator_x=discriminator_x,
                           discriminator_y=discriminator_y,
                           generator_g_optimizer=generator_g_optimizer,
                           generator_f_optimizer=generator_f_optimizer,
                           discriminator_x_optimizer=discriminator_x_optimizer,
                           discriminator_y_optimizer=discriminator_y_optimizer)

ckpt_manager = tf.train.CheckpointManager(ckpt, checkpoint_path, max_to_keep=5)

# if a checkpoint exists, restore the latest checkpoint.
if ckpt_manager.latest_checkpoint:
  ckpt.restore(ckpt_manager.latest_checkpoint)
  print ('Latest checkpoint restored!!')

## Training

In [ ]:
# Проверка формы данных
for normal, weather in train_dataset.take(1):
    print("Normal shape:", normal.shape)
    print("Weather shape:", weather.shape)

Обработка /root/.cache/kagglehub/datasets/nguyenhuuduck16hcm/tt100k/versions/1/tt100k_2021/train:   0%|          | 0/1000 [00:00<?, ?it/s]
Обработка /root/.cache/kagglehub/datasets/njayadithya/rgb-anon-trainvaltest/versions/1/rgb_anon/rain/train: 0it [00:00, ?it/s]
Обработка /root/.cache/kagglehub/datasets/nguyenhuuduck16hcm/tt100k/versions/1/tt100k_2021/train:   0%|          | 0/1000 [00:00<?, ?it/s]


In [ ]:
EPOCHS = 10

In [ ]:
def generate_images(model, test_input):
    prediction = model(test_input)
    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.title('Input (Normal)')
    plt.imshow(test_input[0] * 0.5 + 0.5)

    plt.subplot(1, 2, 2)
    plt.title('Predicted Weather')
    plt.imshow(prediction[0] * 0.5 + 0.5)

    plt.show()

In [ ]:
@tf.function
def train_step(real_x, real_y):
  # persistent is set to True because the tape is used more than
  # once to calculate the gradients.
  with tf.GradientTape(persistent=True) as tape:
    # Generator G translates X -> Y
    # Generator F translates Y -> X.

    fake_y = generator_g(real_x, training=True)
    cycled_x = generator_f(fake_y, training=True)

    fake_x = generator_f(real_y, training=True)
    cycled_y = generator_g(fake_x, training=True)

    # same_x and same_y are used for identity loss.
    same_x = generator_f(real_x, training=True)
    same_y = generator_g(real_y, training=True)

    disc_real_x = discriminator_x(real_x, training=True)
    disc_real_y = discriminator_y(real_y, training=True)

    disc_fake_x = discriminator_x(fake_x, training=True)
    disc_fake_y = discriminator_y(fake_y, training=True)

    # calculate the loss
    gen_g_loss = generator_loss(disc_fake_y)
    gen_f_loss = generator_loss(disc_fake_x)

    total_cycle_loss = calc_cycle_loss(real_x, cycled_x) + calc_cycle_loss(real_y, cycled_y)

    # Total generator loss = adversarial loss + cycle loss
    total_gen_g_loss = gen_g_loss + total_cycle_loss + identity_loss(real_y, same_y)
    total_gen_f_loss = gen_f_loss + total_cycle_loss + identity_loss(real_x, same_x)

    disc_x_loss = discriminator_loss(disc_real_x, disc_fake_x)
    disc_y_loss = discriminator_loss(disc_real_y, disc_fake_y)

  # Calculate the gradients for generator and discriminator
  generator_g_gradients = tape.gradient(total_gen_g_loss,
                                        generator_g.trainable_variables)
  generator_f_gradients = tape.gradient(total_gen_f_loss,
                                        generator_f.trainable_variables)

  discriminator_x_gradients = tape.gradient(disc_x_loss,
                                            discriminator_x.trainable_variables)
  discriminator_y_gradients = tape.gradient(disc_y_loss,
                                            discriminator_y.trainable_variables)

  # Apply the gradients to the optimizer
  generator_g_optimizer.apply_gradients(zip(generator_g_gradients,
                                            generator_g.trainable_variables))

  generator_f_optimizer.apply_gradients(zip(generator_f_gradients,
                                            generator_f.trainable_variables))

  discriminator_x_optimizer.apply_gradients(zip(discriminator_x_gradients,
                                                discriminator_x.trainable_variables))

  discriminator_y_optimizer.apply_gradients(zip(discriminator_y_gradients,
                                                discriminator_y.trainable_variables))

In [ ]:
for epoch in range(EPOCHS):
    start = time.time()

    for n, (input_image, target_image) in enumerate(train_dataset):
        train_step(input_image, target_image)

        if n % 10 == 0:
            print('.', end='')

    # Визуализация прогресса
    clear_output(wait=True)
    sample_normal, sample_weather = next(iter(train_dataset))
    generate_images(generator_g, sample_normal)

    print(f'Epoch {epoch+1} завершена за {time.time()-start:.2f} сек')

Обработка /root/.cache/kagglehub/datasets/nguyenhuuduck16hcm/tt100k/versions/1/tt100k_2021/train:   0%|          | 0/1000 [00:00<?, ?it/s]
Обработка /root/.cache/kagglehub/datasets/njayadithya/rgb-anon-trainvaltest/versions/1/rgb_anon/rain/train: 0it [00:00, ?it/s]
Обработка /root/.cache/kagglehub/datasets/nguyenhuuduck16hcm/tt100k/versions/1/tt100k_2021/train:   0%|          | 0/1000 [00:00<?, ?it/s]


StopIteration: 

## Generate using test dataset

In [ ]:
# Run the trained model on the test dataset
for inp in test_horses.take(5):
  generate_images(generator_g, inp)